<a href="https://colab.research.google.com/github/Nithiasrisaravanan/Agentflow/blob/main/selfpruningnetwork.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install torch torchvision matplotlib


Self-Pruning Neural Network (CIFAR-10)

- Implemented custom PrunableLinear layer
- Used L1 sparsity regularization
- Evaluated for multiple lambda values

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        self.bias = nn.Parameter(torch.zeros(out_features))
        self.gate_scores = nn.Parameter(torch.randn(out_features, in_features))

    def forward(self, x):
        gates = torch.sigmoid(self.gate_scores)
        pruned_weights = self.weight * gates
        return F.linear(x, pruned_weights, self.bias)

In [5]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = PrunableLinear(32*32*3, 256)
        self.fc2 = PrunableLinear(256, 128)
        self.fc3 = PrunableLinear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

In [6]:
def sparsity_loss(model):
    loss = 0
    for m in model.modules():
        if isinstance(m, PrunableLinear):
            gates = torch.sigmoid(m.gate_scores)
            loss += torch.sum(gates)
    return loss

In [7]:
transform = transforms.ToTensor()

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

100%|██████████| 170M/170M [00:23<00:00, 7.35MB/s]


In [10]:
model = Net().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()



lambda_val = 1e-4

for epoch in range(3):
    model.train()
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        cls_loss = criterion(outputs, labels)
        sp_loss = sparsity_loss(model)

        loss = cls_loss + lambda_val * sp_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch", epoch, "done")

Epoch 0 done
Epoch 1 done
Epoch 2 done


In [9]:
def test_accuracy(model):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()

    return 100 * correct / total

def sparsity(model, threshold=1e-2):
    total, zero = 0, 0
    for m in model.modules():
        if isinstance(m, PrunableLinear):
            gates = torch.sigmoid(m.gate_scores)
            total += gates.numel()
            zero += (gates < threshold).sum().item()
    return 100 * zero / total

print("Accuracy:", test_accuracy(model))
print("Sparsity:", sparsity(model), "%")

Accuracy: 38.79
Sparsity: 0.1514966848673947 %


# Self-Pruning Neural Network for CIFAR-10

## 1. Objective

Design a feed-forward neural network that **learns to prune itself during training**. Each weight is associated with a learnable gate that determines whether the connection should be active or removed.

---

## 2. Methodology

### 2.1 Prunable Linear Layer

A custom layer `PrunableLinear` is implemented with:

* Standard **weights** and **bias**
* Additional learnable parameters: **gate_scores**

During the forward pass:

* Gates are computed as: `gates = sigmoid(gate_scores)`
* Effective weights are: `pruned_weights = weights × gates`

This ensures:

* Gates ∈ (0,1)
* Gates close to 0 → connection effectively removed

---

### 2.2 Loss Function

The total loss is defined as:

**Total Loss = Classification Loss + λ × Sparsity Loss**

* Classification Loss: CrossEntropyLoss
* Sparsity Loss: L1 norm of all gate values

---

### 2.3 Why L1 Regularization Encourages Sparsity

L1 regularization penalizes the absolute magnitude of gate values. Since gates are constrained between 0 and 1, minimizing their sum pushes many values toward zero. As a result, less important connections are suppressed, leading to a sparse and efficient network.

---

## 3. Experimental Setup

* Dataset: CIFAR-10
* Model: 3-layer feedforward neural network
* Optimizer: Adam
* Epochs: 3
* Batch Size: 64

---

## 4. Results

| Lambda (λ
